# ML — LR, RF y XGBoost · Nested CV 5×3 con grupos · TF-IDF 10k

Evaluación con `StratifiedGroupKFold`, F1-Macro y separación estricta entre selección interna y evaluación externa. Outer CV = 5 folds; Inner CV = 3 folds.


## 1. Librerías y configuración compartida

Se evita redefinir `DATA_DIR`, `FEATURE_COLS`, `VARIANTES` o `RANDOM_STATE` en secciones distintas. Ajusta `DATA_DIR` si tu notebook no está ubicado junto al directorio `data`.

In [2]:
import os
import json
import time
import joblib
import warnings
import html
import numpy as np
import pandas as pd
import optuna
from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from xgboost import XGBClassifier
import nltk
from nltk.corpus import stopwords

# Lista original del notebook LR/RF. Se usa la misma para los tres modelos.
try:
    STOP_WORDS = stopwords.words("spanish")
except LookupError:
    if not nltk.download("stopwords", quiet=True):
        raise RuntimeError("Instala el corpus NLTK de stopwords para reproducir el TF-IDF original.")
    STOP_WORDS = stopwords.words("spanish")

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "../data"
OUT_DIR = os.path.join(DATA_DIR, "resultados_ml_groupcv_tfidf_lr_rf_10k")
MODELS_DIR = os.path.join(OUT_DIR, "modelos_finales")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
VARIANTES = ["mx", "es", "cu"]
MODELOS_LR_RF = ["LR", "RF"]
PREPROCESAMIENTOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]

N_OUTER = 5
N_INNER = 3
N_TRIALS_NESTED = 50
N_TRIALS_FINAL = 50
N_JOBS_GRID = -1       # Paralelismo de GridSearchCV para LR/RF
N_JOBS_XGB = -1        # Dentro de XGBoost; CV de Optuna secuencial

# Esta corrida fija el PERFIL ORIGINAL de LR/RF en los tres algoritmos.
# Los n-gramas se escogen INTERNAMENTE (1,1) vs (1,2), no con Outer Test.
NGRAMAS_OPCIONES = {"uni": (1, 1), "uni_bi": (1, 2)}
TFIDF_PERFIL = "LR_RF_ORIGINAL_10K_STOPWORDS_ES_NGRAMAS_INNER"
EJECUTAR_TEST_OFICIAL = False  # Cambiar SOLO cuando toda la selección esté cerrada
OOF_REGISTROS = []        # Predicciones independientes del Outer Test (sin datos oficiales de test)

import sklearn
import xgboost
VERSIONES = {
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
    "optuna": optuna.__version__,
}
print("Versiones:", VERSIONES)

print("Algoritmos: LR, RF y XGBoost")
print(f"StratifiedGroup Nested CV: {N_OUTER} outer × {N_INNER} inner; XGBoost {N_TRIALS_NESTED} trials/outer")
print(f"TF-IDF común: {TFIDF_PERFIL} | N-gramas: {NGRAMAS_OPCIONES}")
print(f"Stopwords NLTK español: {len(STOP_WORDS)} | Test oficial: {EJECUTAR_TEST_OFICIAL}")

# ------------------------------------------------------------
# Monitor compacto para la parte larga de LR/RF.
# Una sola salida se actualiza; GridSearchCV permanece con verbose=0.
# ------------------------------------------------------------
class MonitorML:
    def __init__(self, total_bloques):
        self.total = int(total_bloques)
        self.completados = 0
        self.inicio = time.monotonic()
        self.etapa = "Inicializando LR/RF"
        self.handle = display(HTML("<b>ML — preparando GridSearch...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        minutos = (time.monotonic() - self.inicio) / 60
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / self.completados * faltan:.0f} min"
            if self.completados >= 2 else "calculando..."
        )
        panel = (
            '<div style="font-family:system-ui;padding:12px;border:1px solid #777;'
            'border-radius:8px;max-width:800px">'
            '<b>ML — seguimiento en vivo</b><br>'
            f'Etapa: <b>{html.escape(self.etapa)}</b><br>'
            f'Búsquedas GridSearch completadas: {self.completados}/{self.total} '
            f'({100*self.completados/max(1,self.total):.1f}%)<br>'
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f'Tiempo: {minutos:.1f} min | ETA orientativo: {eta}'
            '</div>'
        )
        self.handle.update(HTML(panel))

    def iniciar(self, etapa):
        self.mostrar(etapa=etapa, forzar=True)

    def completar(self, etapa):
        self.completados += 1
        self.mostrar(etapa=etapa, forzar=True)


Versiones: {'numpy': '1.26.4', 'pandas': '3.0.3', 'sklearn': '1.8.0', 'xgboost': '3.2.0', 'optuna': '5.0.0'}
Algoritmos: LR, RF y XGBoost
StratifiedGroup Nested CV: 5 outer × 3 inner; XGBoost 50 trials/outer
TF-IDF común: LR_RF_ORIGINAL_10K_STOPWORDS_ES_NGRAMAS_INNER | N-gramas: {'uni': (1, 1), 'uni_bi': (1, 2)}
Stopwords NLTK español: 313 | Test oficial: False


## 2. Integridad de los CSV y validación por grupos

Los CSV de train ya llegan **descontaminados desde `01_preprocesamiento_ML_CORREGIDO.ipynb`**.

Este notebook no vuelve a comparar ni eliminar filas contra el test oficial.

Aquí se verifica únicamente:

- alineación entre `normal`, `stem` y `lemma`;
- IDs/etiquetas/rasgos consistentes entre las tres representaciones;
- construcción de `GROUP_ID` usando original + normal + stem + lemma;
- ausencia de grupos compartidos entre train y validación en Outer e Inner CV.

`StratifiedGroupKFold` mantiene cada grupo completo dentro de un solo lado del split.


In [3]:
# ============================================================
# 2. CARGA, INTEGRIDAD Y GRUPOS SIN FUGA ENTRE FOLDS
# ============================================================
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

GRUPOS_TRAIN = {}        # Variante -> np.array; solo contiene información del TRAIN
AUDITORIA_GRUPOS = []
AUDITORIA_CONFLICTOS = []
AUDITORIA_PARTICIONES = []


def cargar_split(variante, prep, split="train"):
    if variante not in VARIANTES or prep not in PREPROCESAMIENTOS:
        raise ValueError(f"Variante/preprocessing no reconocido: {variante}/{prep}")
    ruta = os.path.join(DATA_DIR, f"{split}_clean{PREPROCESAMIENTOS[prep]}_{variante}.csv")
    if not os.path.isfile(ruta):
        raise FileNotFoundError(f"Falta el CSV: {ruta}")
    # Preservar los IDs originales: no forzarles una conversión numérica.
    return pd.read_csv(ruta, dtype={"ID": "string"}).reset_index(drop=True)


def preparar_xy(df):
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X.reset_index(drop=True), y


def texto_clave(serie):
    # Misma normalización de COMPARACIÓN que la auditoría previa; no modifica el texto del modelo.
    return serie.fillna("").astype(str).str.casefold().str.strip()


def construir_grupos(variante, dfs_por_prep, y):
    """Componentes conexos: agrupa un mismo texto tras normal, stem O lemma.
    No deduplica, no usa las etiquetas para construir grupos y conserva rasgos originales.
    Añade coincidencias de MESSAGE original exacto, si está disponible.
    """
    n = len(y)
    parent = list(range(n))
    size = [1] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]

    for prep, df in dfs_por_prep.items():
        primero = {}
        for i, clave in enumerate(texto_clave(df["MESSAGE_CLEAN"])):
            if not clave:
                raise ValueError(f"{variante}/{prep}: texto limpio vacío en fila {i}; revisar antes de agrupar")
            if clave in primero:
                union(primero[clave], i)
            else:
                primero[clave] = i
    # Si dos textos originales son idénticos, deben permanecer en el mismo grupo.
    df_ref = dfs_por_prep["normal"]
    if "MESSAGE" in df_ref:
        primero_raw = {}
        for i, clave in enumerate(texto_clave(df_ref["MESSAGE"])):
            if not clave:
                continue
            if clave in primero_raw:
                union(primero_raw[clave], i)
            else:
                primero_raw[clave] = i

    miembros = {}
    for i in range(n):
        miembros.setdefault(find(i), []).append(i)
    # Id de grupo basado en el menor índice original; determinista entre ambos notebooks.
    ids = {ra: min(idx) for ra, idx in miembros.items()}
    grupos = np.array([ids[find(i)] for i in range(n)], dtype=int)
    duplicados = {ids[ra]: idx for ra, idx in miembros.items() if len(idx) > 1}
    conflictivos = [(g, idx) for g, idx in duplicados.items() if y.iloc[idx].nunique() > 1]
    if conflictivos:
        warnings.warn(
            f"{variante.upper()}: {len(conflictivos)} grupo(s) con etiquetas contradictorias. "
            "Se conservan las etiquetas originales y se mantienen juntos todos sus registros. "
            "Revisar manualmente antes de interpretar resultados.", stacklevel=2)
    for g, idx in conflictivos:
        for i in idx:
            AUDITORIA_CONFLICTOS.append({
                "variante": variante, "grupo": int(g), "fila_0base": int(i),
                "id_original": str(df_ref.iloc[i]["ID"]) if "ID" in df_ref else "",
                "etiqueta": int(y.iloc[i]),
                "texto_original": str(df_ref.iloc[i]["MESSAGE"]) if "MESSAGE" in df_ref else "",
                "texto_normal": str(df_ref.iloc[i]["MESSAGE_CLEAN"]),
            })
    AUDITORIA_GRUPOS.append({
        "variante": variante, "n_train": n, "n_grupos": len(miembros),
        "n_grupos_multiples": len(duplicados),
        "n_filas_en_grupos_multiples": sum(map(len, duplicados.values())),
        "tamano_grupo_max": max(map(len, miembros.values())),
        "n_grupos_etiquetas_contradictorias": len(conflictivos),
        "criterio": "union_texto_normal_stem_lemma_y_original_exacto",
    })
    return grupos


def cargar_todos_preps(variante, split="train"):
    dfs = {p: cargar_split(variante, p, split=split) for p in PREPROCESAMIENTOS}
    ref = dfs["normal"]
    X_ref, y_ref = preparar_xy(ref)
    datos = {"normal": X_ref}
    for prep, df in dfs.items():
        if prep == "normal":
            continue
        X, y = preparar_xy(df)
        if len(df) != len(ref) or not y.equals(y_ref):
            raise ValueError(f"{variante}/{prep}: diferente número de filas o etiquetas")
        for c in ["ID", "MESSAGE"]:
            if c in ref.columns and c not in df.columns:
                raise ValueError(f"{variante}/{prep}: falta columna de alineación {c}")
            if c in ref.columns and not ref[c].astype(str).equals(df[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: columna {c} desalineada")
        if not X[FEATURE_COLS].equals(X_ref[FEATURE_COLS]):
            raise ValueError(f"{variante}/{prep}: rasgos lingüísticos no alineados")
        datos[prep] = X
    if split == "train" and variante not in GRUPOS_TRAIN:
        GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)
    return datos, y_ref


def grupos_train(variante):
    if variante not in GRUPOS_TRAIN:
        cargar_todos_preps(variante, split="train")
    return GRUPOS_TRAIN[variante]


def verificar_particion(grupos, a, b, variante, nivel, outer, inner=None):
    solapamiento = set(grupos[a]) & set(grupos[b])
    if solapamiento:
        raise AssertionError(f"FUGA: {variante}/{nivel}/{outer}/{inner} grupos compartidos: {solapamiento}")
    AUDITORIA_PARTICIONES.append({
        "variante": variante, "nivel": nivel, "outer_fold": outer,
        "inner_fold": inner, "n_train": len(a), "n_validacion": len(b),
        "n_grupos_compartidos": 0,
    })


def crear_splits_outer(y, grupos, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "outer", fold)
    return splits


def crear_splits_inner(y, grupos, outer_fold, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE+outer_fold)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "inner", outer_fold, fold)
    return splits


def crear_splits_final(y, grupos, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "final", 0, fold)
    return splits


# ============================================================
# AUDITORÍA PREVIA — SOLO TRAIN Y FOLDS
# ============================================================
for v in VARIANTES:
    _, etiquetas = cargar_todos_preps(v, split="train")
    gv = grupos_train(v)

    for o, (a, b) in enumerate(
        crear_splits_outer(etiquetas, gv, v),
        1,
    ):
        crear_splits_inner(
            etiquetas.iloc[a].reset_index(drop=True),
            gv[a],
            o,
            v,
        )

    crear_splits_final(etiquetas, gv, v)


df_auditoria_grupos = pd.DataFrame(AUDITORIA_GRUPOS)

df_auditoria_particiones = (
    pd.DataFrame(AUDITORIA_PARTICIONES)
    .drop_duplicates()
)

df_auditoria_conflictos = pd.DataFrame(AUDITORIA_CONFLICTOS)

print("AUDITORÍA PREVIA — TRAIN Y GROUPED CV")
display(df_auditoria_grupos)

print("\nSolapamientos entre folds (todos deben ser 0):")
display(
    df_auditoria_particiones
    .groupby(
        ["variante", "nivel"],
        as_index=False,
    )["n_grupos_compartidos"]
    .max()
)

pd.DataFrame(AUDITORIA_GRUPOS).to_csv(
    os.path.join(OUT_DIR, "auditoria_grupos.csv"),
    index=False,
)

df_auditoria_particiones.to_csv(
    os.path.join(OUT_DIR, "auditoria_folds.csv"),
    index=False,
)

df_auditoria_conflictos.to_csv(
    os.path.join(OUT_DIR, "auditoria_etiquetas_contradictorias.csv"),
    index=False,
)

print("\n[PRE-FLIGHT][ML] Train alineado + Grouped CV: OK")


C:\Users\I21309\AppData\Local\Temp\ipykernel_25988\1989180649.py:138: UserWarning: ES: 1 grupo(s) con etiquetas contradictorias. Se conservan las etiquetas originales y se mantienen juntos todos sus registros. Revisar manualmente antes de interpretar resultados.
  GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)


AUDITORÍA PREVIA — TRAIN Y GROUPED CV


,variante,n_train,n_grupos,n_grupos_multiples,n_filas_en_grupos_multiples,tamano_grupo_max,n_grupos_etiquetas_contradictorias,criterio
0,mx,2399,2393,3,9,5,0,union_texto_normal_stem_lemma_y_original_exacto
1,es,2397,2289,45,153,17,1,union_texto_normal_stem_lemma_y_original_exacto
2,cu,2400,2389,11,22,2,0,union_texto_normal_stem_lemma_y_original_exacto



Solapamientos entre folds (todos deben ser 0):


,variante,nivel,n_grupos_compartidos
0,cu,final,0
1,cu,inner,0
2,cu,outer,0
3,es,final,0
4,es,inner,0
5,es,outer,0
6,mx,final,0
7,mx,inner,0
8,mx,outer,0



[PRE-FLIGHT][ML] Train alineado + Grouped CV: OK


## 3. TF-IDF original LR/RF, común a LR, RF y XGBoost

Se reproduce `TfidfVectorizer(stop_words=STOP_WORDS, max_features=10000)` del notebook original de LR/RF, **incluyendo** la comparación original `(1,1)` y `(1,2)` dentro del Inner CV. Todos los algoritmos tienen las mismas opciones de representación; cada uno puede elegir un rango de n-gramas diferente sin afectar la evaluación independiente del Outer Fold.

**Nota sobre stopwords:** se conservan literalmente para este experimento controlado, pero su lista puede incluir términos de negación útiles en sarcasmo. Si posteriormente se evalúa otra lista, tendrá que ser otra corrida separada.


In [4]:
def crear_preprocesador_tfidf(ngram_range=(1, 1)):
    """Exactamente el TF-IDF base original LR/RF y sus 9 rasgos."""
    tfidf = TfidfVectorizer(
        stop_words=STOP_WORDS,
        max_features=10000,
        ngram_range=ngram_range,
    )
    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )

# Única representación compartida por todos los modelos.
crear_preprocesador_lr_rf = crear_preprocesador_tfidf
crear_preprocesador_xgb = crear_preprocesador_tfidf

print("TF-IDF original LR/RF: stopwords ES, max_features=10000, ngramas=(1,1)/(1,2) elegidos en Inner CV.")


TF-IDF original LR/RF: stopwords ES, max_features=10000, ngramas=(1,1)/(1,2) elegidos en Inner CV.


## 4. LR y RF — GridSearchCV con TF-IDF original y preprocessing dentro del Inner CV

Se conservan las rejillas de hiperparámetros de los clasificadores del notebook anterior. Se añade para ambos el parámetro **original** `features__tfidf__ngram_range` con `(1,1)` y `(1,2)`. En cada Outer Fold, `normal/stem/lemma`, n-gramas y parámetros del clasificador se deciden exclusivamente con Inner CV; Outer Test permanece aislado.


In [5]:
def crear_pipeline_lr_rf(modelo):
    if modelo == "LR":
        clf = LogisticRegression(
            max_iter=10000, random_state=RANDOM_STATE, class_weight="balanced"
        )
    elif modelo == "RF":
        clf = RandomForestClassifier(
            random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1
        )
    else:
        raise ValueError(f"Modelo LR/RF no reconocido: {modelo}")
    return Pipeline(steps=[("features", crear_preprocesador_lr_rf()), ("clf", clf)])


def grid_lr_rf(modelo):
    if modelo == "LR":
        grid = {"clf__C": [0.1, 1.0, 10], "clf__solver": ["liblinear"]}
    elif modelo == "RF":
        grid = {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 5],
        }
    else:
        raise ValueError(modelo)
    grid["features__tfidf__ngram_range"] = [(1, 1), (1, 2)]
    return grid


def buscar_lr_rf_train(X_por_prep, y, modelo, inner_splits, monitor=None, variante=None, outer_fold=None):
    """Selecciona preprocessing + hiperparámetros EXCLUSIVAMENTE por puntuación Inner CV."""
    mejor = None
    candidatos = []
    for prep, X in X_por_prep.items():
        etiqueta = (
            f"[{str(variante).upper()}][{modelo}][Outer {outer_fold}/{N_OUTER}] "
            f"GridSearch | prep={prep}"
        )
        if monitor is not None:
            monitor.iniciar(etiqueta)
        search = GridSearchCV(
            estimator=crear_pipeline_lr_rf(modelo),
            param_grid=grid_lr_rf(modelo),
            cv=inner_splits,
            scoring="f1_macro",
            n_jobs=N_JOBS_GRID,
            refit=True,
            error_score="raise",
        )
        search.fit(X, y)
        if monitor is not None:
            monitor.completar(etiqueta + " | completado")
        scores_folds = [
            search.cv_results_[f"split{i}_test_score"][search.best_index_]
            for i in range(len(inner_splits))
        ]
        desviacion = float(np.std(scores_folds, ddof=1))
        candidato = {
            "prep": prep,
            "inner_f1": float(search.best_score_),
            "inner_std": desviacion,
            "params": search.best_params_,
            "estimator": search.best_estimator_,
        }
        candidatos.append({k: v for k, v in candidato.items() if k != "estimator"})
        if (mejor is None or candidato["inner_f1"] > mejor["inner_f1"]
            or (np.isclose(candidato["inner_f1"], mejor["inner_f1"])
                and candidato["inner_std"] < mejor["inner_std"])):
            mejor = candidato
    return mejor, candidatos


def nested_cv_lr_rf_variante(variante, modelo, monitor=None):
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    detalle, candidatos_inner = [], []
    grupos = grupos_train(variante)
    for outer_fold, (i_train, i_test) in enumerate(crear_splits_outer(y, grupos, variante), start=1):
        y_train, y_test = y.iloc[i_train].reset_index(drop=True), y.iloc[i_test].reset_index(drop=True)
        X_train = {p: X.iloc[i_train].reset_index(drop=True) for p, X in X_por_prep.items()}
        X_outer = {p: X.iloc[i_test].reset_index(drop=True) for p, X in X_por_prep.items()}
        mejor, candidatos = buscar_lr_rf_train(
            X_train, y_train, modelo,
            crear_splits_inner(y_train, grupos[i_train], outer_fold, variante),
            monitor=monitor, variante=variante, outer_fold=outer_fold,
        )
        for c in candidatos:
            candidatos_inner.append({
                "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
                "preprocessing": c["prep"], "inner_f1_macro": c["inner_f1"],
                "inner_f1_std": c["inner_std"], "best_params": c["params"],
            })
        prep = mejor["prep"]
        # search.best_estimator_ ya es un refit nuevo sobre TODO este outer_train.
        estimador = mejor["estimator"]
        pred_train = estimador.predict(X_train[prep])
        pred_outer = estimador.predict(X_outer[prep])
        train_f1 = f1_score(y_train, pred_train, average="macro")
        outer_f1 = f1_score(y_test, pred_outer, average="macro")
        # Cada registro figura exactamente una vez como Outer Test en una CV 5-fold.
        OOF_REGISTROS.append(pd.DataFrame({
            "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
            "idx_train_oficial": i_test, "y_true": y_test.to_numpy(),
            "y_pred": np.asarray(pred_outer),
        }))
        f1_clase = f1_score(y_test, pred_outer, labels=[0, 1], average=None, zero_division=0)
        detalle.append({
            "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
            "outer_f1_clase_0": float(f1_clase[0]), "outer_f1_clase_1": float(f1_clase[1]),
            "best_preprocessing": prep, "train_f1_macro": float(train_f1),
            "inner_best_f1": mejor["inner_f1"], "inner_best_std": mejor["inner_std"],
            "outer_f1_macro": float(outer_f1),
            "generalization_gap": float(train_f1 - outer_f1),
            "best_params": mejor["params"],
        })
        print(
            f"[{variante.upper()}][{modelo}][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={prep} | F1_inner={mejor['inner_f1']:.4f}±{mejor['inner_std']:.4f} | "
            f"F1_train={train_f1:.4f} | F1_outer={outer_f1:.4f} | gap={train_f1 - outer_f1:.4f}"
        )
    return pd.DataFrame(detalle), pd.DataFrame(candidatos_inner)

### 4.1 Ejecutar Nested CV de LR y RF (solo train)

Esta celda es computacionalmente costosa: evalúa las combinaciones de cada grid sobre los 5 Inner folds y las 3 alternativas de preprocessing en cada uno de los 5 Outer folds.

In [6]:
total_bloques_lr_rf = len(VARIANTES) * len(MODELOS_LR_RF) * N_OUTER * len(PREPROCESAMIENTOS)
monitor_lr_rf = MonitorML(total_bloques=total_bloques_lr_rf)

resultados_nested_lr_rf = {}
registros_lr_rf = []
candidatos_inner_lr_rf = []

for variante in VARIANTES:
    for modelo in MODELOS_LR_RF:
        detalle, candidatos = nested_cv_lr_rf_variante(variante, modelo, monitor=monitor_lr_rf)
        resultados_nested_lr_rf[(variante, modelo)] = detalle
        candidatos_inner_lr_rf.append(candidatos)
        registros_lr_rf.append({
            "variante": variante,
            "modelo": modelo,
            "f1_train_mean": detalle["train_f1_macro"].mean(),
            "f1_nested_mean": detalle["outer_f1_macro"].mean(),
            "f1_nested_std": detalle["outer_f1_macro"].std(ddof=1),
            "gap_mean": detalle["generalization_gap"].mean(),
        })

df_nested_lr_rf = pd.DataFrame(registros_lr_rf)
df_candidatos_inner_lr_rf = pd.concat(candidatos_inner_lr_rf, ignore_index=True)
monitor_lr_rf.mostrar("LR/RF completado", forzar=True)
print("\n[ML][LR/RF] RESUMEN NESTED CV")
display(df_nested_lr_rf.round(4))

[MX][LR][Outer 1/5] FINALIZADO | prep=lemma | F1_inner=0.6421±0.0206 | F1_train=0.8932 | F1_outer=0.6291 | gap=0.2641
[MX][LR][Outer 2/5] FINALIZADO | prep=normal | F1_inner=0.6324±0.0098 | F1_train=0.9087 | F1_outer=0.6511 | gap=0.2576
[MX][LR][Outer 3/5] FINALIZADO | prep=normal | F1_inner=0.6548±0.0036 | F1_train=0.9153 | F1_outer=0.6674 | gap=0.2479
[MX][LR][Outer 4/5] FINALIZADO | prep=lemma | F1_inner=0.6440±0.0211 | F1_train=0.8856 | F1_outer=0.6403 | gap=0.2453
[MX][LR][Outer 5/5] FINALIZADO | prep=stem | F1_inner=0.6457±0.0086 | F1_train=0.8714 | F1_outer=0.6371 | gap=0.2343
[MX][RF][Outer 1/5] FINALIZADO | prep=stem | F1_inner=0.6309±0.0104 | F1_train=0.8904 | F1_outer=0.6501 | gap=0.2403
[MX][RF][Outer 2/5] FINALIZADO | prep=normal | F1_inner=0.6264±0.0086 | F1_train=0.8134 | F1_outer=0.5993 | gap=0.2141
[MX][RF][Outer 3/5] FINALIZADO | prep=normal | F1_inner=0.6400±0.0242 | F1_train=0.8018 | F1_outer=0.6319 | gap=0.1699
[MX][RF][Outer 4/5] FINALIZADO | prep=normal | F1_inne

,variante,modelo,f1_train_mean,f1_nested_mean,f1_nested_std,gap_mean
0,mx,LR,0.8948,0.6450,0.0148,0.2498
1,mx,RF,0.8221,0.6307,0.0197,0.1913
2,es,LR,0.9230,0.7024,0.0157,0.2205
3,es,RF,0.9446,0.6959,0.0298,0.2487
4,cu,LR,0.9275,0.6586,0.0172,0.2689
5,cu,RF,0.8425,0.6608,0.0305,0.1817


## 5. XGBoost — Optuna 50 trials y mismo TF-IDF original LR/RF

El espacio regularizado de **XGBoost no cambia**. Cada trial ahora propone también `ngramas ∈ {uni, uni_bi}` además de `normal/stem/lemma`. Las **primeras seis propuestas** cubren las seis combinaciones de preprocessing × n-gramas; los 44 trials restantes los elige Optuna. Son **50 trials EN TOTAL por Outer Fold**, no 50 por combinación. TF-IDF sigue dentro del Pipeline, que se ajusta desde cero en cada fold interno.


In [7]:
# Alias compartido: mismo TF-IDF original LR/RF para LR, RF y XGBoost.
crear_preprocesador = crear_preprocesador_tfidf

In [8]:
# ============================================================
# 4. ESPACIO DE BÚSQUEDA PARA OPTUNA
# ============================================================

def sugerir_params_xgb(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 15),
        "subsample": trial.suggest_float("subsample", 0.55, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.90),
        "gamma": trial.suggest_float("gamma", 0.5, 8.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 2.5),
    }


def crear_xgboost(params, random_state=RANDOM_STATE):
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=N_JOBS_XGB,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE, ngramas="uni"):
    if ngramas not in NGRAMAS_OPCIONES:
        raise ValueError(f"Opción de n-gramas no reconocida: {ngramas}")
    return Pipeline(
        steps=[
            ("features", crear_preprocesador(ngram_range=NGRAMAS_OPCIONES[ngramas])),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )

In [9]:
def nested_cv_variante(variante, n_trials=N_TRIALS_NESTED):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    grupos = grupos_train(variante)

    resultados_outer = []
    studies_outer = {}
    X_ref = X_por_prep["normal"]

    for outer_fold, (idx_train, idx_test) in enumerate(
        crear_splits_outer(y, grupos, variante), start=1
    ):
        print(f"\n[{variante.upper()}][XGBoost][Outer {outer_fold}/{N_OUTER}] Optuna TPE")

        y_outer_train = y.iloc[idx_train].reset_index(drop=True)
        y_outer_test = y.iloc[idx_test].reset_index(drop=True)

        X_outer_train = {
            prep: X.iloc[idx_train].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }
        X_outer_test = {
            prep: X.iloc[idx_test].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }

        inner_splits = crear_splits_inner(y_outer_train, grupos[idx_train], outer_fold, variante)

        def objective(trial):
            prep = trial.suggest_categorical(
                "preprocessing", list(PREPROCESAMIENTOS.keys())
            )
            ngramas = trial.suggest_categorical("ngramas", list(NGRAMAS_OPCIONES))
            params = sugerir_params_xgb(trial)
            pipeline = crear_pipeline(
                params, random_state=RANDOM_STATE + outer_fold, ngramas=ngramas
            )

            scores = cross_val_score(
                estimator=pipeline,
                X=X_outer_train[prep],
                y=y_outer_train,
                cv=inner_splits,
                scoring="f1_macro",
                n_jobs=1,
            )

            trial.set_user_attr("inner_f1_std", float(scores.std(ddof=1)))
            return float(scores.mean())

        sampler = optuna.samplers.TPESampler(
            seed=RANDOM_STATE + outer_fold
        )
        study = optuna.create_study(
            direction="maximize", sampler=sampler
        )
        # Se garantiza cobertura inicial de las 3 versiones × las 2 opciones de n-gramas.
        semillas = [(prep, ng) for prep in PREPROCESAMIENTOS for ng in NGRAMAS_OPCIONES]
        if n_trials >= len(semillas):
            for prep_inicial, ngramas_inicial in semillas:
                study.enqueue_trial({"preprocessing": prep_inicial, "ngramas": ngramas_inicial})
        study.optimize(
            objective,
            n_trials=n_trials,
            show_progress_bar=True,
        )
        studies_outer[outer_fold] = study

        best_all = study.best_params.copy()
        best_prep = best_all.pop("preprocessing")
        best_ngramas = best_all.pop("ngramas")
        best_params = best_all

        modelo_outer = crear_pipeline(
            best_params,
            random_state=RANDOM_STATE + outer_fold,
            ngramas=best_ngramas,
        )
        modelo_outer.fit(
            X_outer_train[best_prep], y_outer_train
        )

        pred_train = modelo_outer.predict(X_outer_train[best_prep])
        pred_outer = modelo_outer.predict(X_outer_test[best_prep])

        f1_train = f1_score(y_outer_train, pred_train, average="macro")
        f1_outer = f1_score(y_outer_test, pred_outer, average="macro")
        gap = f1_train - f1_outer

        OOF_REGISTROS.append(pd.DataFrame({
            "variante": variante, "modelo": "XGBoost", "outer_fold": outer_fold,
            "idx_train_oficial": idx_test, "y_true": y_outer_test.to_numpy(),
            "y_pred": np.asarray(pred_outer),
        }))
        f1_clase = f1_score(y_outer_test, pred_outer, labels=[0, 1], average=None, zero_division=0)
        resultados_outer.append({
            "outer_f1_clase_0": float(f1_clase[0]),
            "outer_f1_clase_1": float(f1_clase[1]),
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": best_prep,
            "best_ngramas": best_ngramas,
            "train_f1_macro": f1_train,
            "inner_best_f1": study.best_value,
            "inner_best_std": study.best_trial.user_attrs.get(
                "inner_f1_std", np.nan
            ),
            "outer_f1_macro": f1_outer,
            "generalization_gap": gap,
            "best_params": best_params,
        })

        print(
            f"[{variante.upper()}][XGBoost][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={best_prep} | ngramas={best_ngramas} | "
            f"F1_inner={study.best_value:.4f}±{study.best_trial.user_attrs.get('inner_f1_std', np.nan):.4f} | "
            f"F1_train={f1_train:.4f} | F1_outer={f1_outer:.4f} | gap={gap:.4f}"
        )

    df_outer = pd.DataFrame(resultados_outer)
    resumen = {
        "variante": variante,
        "f1_train_mean": df_outer["train_f1_macro"].mean(),
        "f1_nested_mean": df_outer["outer_f1_macro"].mean(),
        "f1_nested_std": df_outer["outer_f1_macro"].std(ddof=1),
        "f1_nested_min": df_outer["outer_f1_macro"].min(),
        "f1_nested_max": df_outer["outer_f1_macro"].max(),
        "gap_mean": df_outer["generalization_gap"].mean(),
    }
    return df_outer, resumen, studies_outer

### 5.1 Ejecutar Nested CV de XGBoost (solo train)

**Costo aproximado:** 3 variantes × 5 Outer × 50 trials × 5 Inner = 3 750 ajustes de XGBoost, sin contar refits. Se realizan sobre el train oficial; esta sección no carga el test.

In [ ]:
# ============================================================
# 6. EJECUTAR NESTED CV PARA LAS 3 VARIANTES
# ============================================================

nested_detalle = {}
nested_resumen = []
nested_studies = {}

for variante in VARIANTES:
    detalle, resumen, studies = nested_cv_variante(
        variante, n_trials=N_TRIALS_NESTED
    )
    nested_detalle[variante] = detalle
    nested_resumen.append(resumen)
    nested_studies[variante] = studies


df_nested_resumen = pd.DataFrame(nested_resumen)

print("\nRESULTADOS NESTED CV 5×3")
display(
    df_nested_resumen.style.format({
        "f1_train_mean": "{:.4f}",
        "f1_nested_mean": "{:.4f}",
        "f1_nested_std": "{:.4f}",
        "f1_nested_min": "{:.4f}",
        "f1_nested_max": "{:.4f}",
        "gap_mean": "{:.4f}",
    })
)


[MX][XGBoost][Outer 1/5] Optuna TPE


  0%|          | 0/50 [00:00<?, ?it/s]

[MX][XGBoost][Outer 1/5] FINALIZADO | prep=stem | ngramas=uni | F1_inner=0.6155±0.0117 | F1_train=0.8893 | F1_outer=0.6046 | gap=0.2847

[MX][XGBoost][Outer 2/5] Optuna TPE


  0%|          | 0/50 [00:00<?, ?it/s]

[MX][XGBoost][Outer 2/5] FINALIZADO | prep=stem | ngramas=uni | F1_inner=0.5970±0.0020 | F1_train=0.6333 | F1_outer=0.5981 | gap=0.0352

[MX][XGBoost][Outer 3/5] Optuna TPE


...

  0%|          | 0/50 [00:00<?, ?it/s]

[ES][XGBoost][Outer 4/5] FINALIZADO | prep=stem | ngramas=uni | F1_inner=0.6871±0.0222 | F1_train=0.7241 | F1_outer=0.6997 | gap=0.0245

[ES][XGBoost][Outer 5/5] Optuna TPE


  0%|          | 0/50 [00:00<?, ?it/s]

[ES][XGBoost][Outer 5/5] FINALIZADO | prep=stem | ngramas=uni | F1_inner=0.6961±0.0091 | F1_train=0.7511 | F1_outer=0.6735 | gap=0.0776

[CU][XGBoost][Outer 1/5] Optuna TPE


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# Detalle de los 5 Outer folds por variante

for variante in VARIANTES:
    print(f"\nDETALLE OUTER — {variante.upper()}")
    display(
        nested_detalle[variante][[
            "outer_fold",
            "best_preprocessing",
            "best_ngramas",
            "train_f1_macro",
            "inner_best_f1",
            "inner_best_std",
            "outer_f1_macro",
            "generalization_gap",
        ]].style.format({
            "train_f1_macro": "{:.4f}",
            "inner_best_f1": "{:.4f}",
            "inner_best_std": "{:.4f}",
            "outer_f1_macro": "{:.4f}",
            "generalization_gap": "{:.4f}",
        })
    )

    print("\nMejor combinación por Outer Fold:")
    for _, row in nested_detalle[variante].iterrows():
        print(
            f"  Fold {int(row['outer_fold'])}: "
            f"prep={row['best_preprocessing']} | ngramas={row['best_ngramas']} | "
            f"params={row['best_params']}"
        )

    print("\nFrecuencia de preprocessing ganador:")
    print(nested_detalle[variante]["best_preprocessing"].value_counts())
    print("Frecuencia de n-gramas ganadores:")
    print(nested_detalle[variante]["best_ngramas"].value_counts())


DETALLE OUTER — MX


,outer_fold,best_preprocessing,best_ngramas,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,stem,uni,0.8280,0.6056,0.0167,0.6066,0.2214
1,2,stem,uni,0.7876,0.6075,0.0350,0.6452,0.1423
2,3,normal,uni,0.7878,0.6257,0.0219,0.5578,0.2300
3,4,stem,uni,0.6901,0.6068,0.0268,0.6261,0.0640
4,5,normal,uni,0.6264,0.6082,0.0255,0.5856,0.0408



Mejor combinación por Outer Fold:
  Fold 1: prep=stem | ngramas=uni | params={'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.043639689845318415, 'min_child_weight': 2, 'subsample': 0.7148569283275314, 'colsample_bytree': 0.6955032024403796, 'gamma': 1.5856736251928039, 'reg_alpha': 0.19003802367439404, 'reg_lambda': 0.6190863822589344, 'scale_pos_weight': 1.9579296055513413}
  Fold 2: prep=stem | ngramas=uni | params={'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.0553808210147886, 'min_child_weight': 3, 'subsample': 0.6705146951557133, 'colsample_bytree': 0.6183002580460784, 'gamma': 0.764968271070108, 'reg_alpha': 0.9931249154773975, 'reg_lambda': 0.1253906180014041, 'scale_pos_weight': 1.661750925937473}
  Fold 3: prep=normal | ngramas=uni | params={'n_estimators': 650, 'max_depth': 6, 'learning_rate': 0.013903488996358949, 'min_child_weight': 2, 'subsample': 0.6440510603628905, 'colsample_bytree': 0.5916791063134984, 'gamma': 1.307066130336734, 'reg_alpha': 0.001

,outer_fold,best_preprocessing,best_ngramas,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,stem,uni,0.7316,0.6929,0.0217,0.6886,0.0430
1,2,lemma,uni_bi,0.7475,0.7009,0.0190,0.6657,0.0818
2,3,normal,uni,0.8230,0.6935,0.0281,0.6685,0.1546
3,4,stem,uni,0.8674,0.6890,0.0272,0.7099,0.1575
4,5,normal,uni,0.7402,0.6937,0.0129,0.6886,0.0516



Mejor combinación por Outer Fold:
  Fold 1: prep=stem | ngramas=uni | params={'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.013101268919773158, 'min_child_weight': 8, 'subsample': 0.8329101372894524, 'colsample_bytree': 0.7359523274253197, 'gamma': 7.635249198936578, 'reg_alpha': 0.001372501841222617, 'reg_lambda': 0.34018780622188, 'scale_pos_weight': 1.6843253789375556}
  Fold 2: prep=lemma | ngramas=uni_bi | params={'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.026756792630276614, 'min_child_weight': 5, 'subsample': 0.8621746951985744, 'colsample_bytree': 0.5234354113061952, 'gamma': 6.536221688356087, 'reg_alpha': 0.1419560197684238, 'reg_lambda': 1.3769517137542973, 'scale_pos_weight': 1.3785267824047707}
  Fold 3: prep=normal | ngramas=uni | params={'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.028249372273012368, 'min_child_weight': 2, 'subsample': 0.5594194438230156, 'colsample_bytree': 0.7157919936907968, 'gamma': 4.404681912077528, 'reg_alpha': 

,outer_fold,best_preprocessing,best_ngramas,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,normal,uni,0.6875,0.6533,0.0303,0.6375,0.0500
1,2,stem,uni_bi,0.8339,0.6566,0.0103,0.6471,0.1868
2,3,stem,uni_bi,0.7937,0.6629,0.0149,0.6532,0.1405
3,4,lemma,uni,0.7956,0.6556,0.0215,0.6441,0.1515
4,5,lemma,uni,0.7354,0.6458,0.0159,0.6435,0.0919



Mejor combinación por Outer Fold:
  Fold 1: prep=normal | ngramas=uni | params={'n_estimators': 250, 'max_depth': 2, 'learning_rate': 0.012145880093147213, 'min_child_weight': 2, 'subsample': 0.7924592276418388, 'colsample_bytree': 0.571923704962879, 'gamma': 5.48072579127046, 'reg_alpha': 0.001803432229808866, 'reg_lambda': 0.18096853629124918, 'scale_pos_weight': 2.2471850290754856}
  Fold 2: prep=stem | ngramas=uni_bi | params={'n_estimators': 450, 'max_depth': 5, 'learning_rate': 0.06356451422238978, 'min_child_weight': 2, 'subsample': 0.878845023171697, 'colsample_bytree': 0.6949158605577016, 'gamma': 7.642276994354405, 'reg_alpha': 0.007951608314056518, 'reg_lambda': 0.18455632294616425, 'scale_pos_weight': 1.9247641903135144}
  Fold 3: prep=stem | ngramas=uni_bi | params={'n_estimators': 650, 'max_depth': 4, 'learning_rate': 0.010300045354980825, 'min_child_weight': 2, 'subsample': 0.6192069325659431, 'colsample_bytree': 0.7540218386885148, 'gamma': 0.7574435922835878, 'reg_alp

## 6. Comparación conjunta y diagnóstico — solo TRAIN oficial

Los tres modelos utilizan los mismos Outer/Inner folds, **exactamente las mismas opciones de TF-IDF original LR/RF** (10 000 términos, stopwords NLTK, 1–1 o 1–2 gramas), nueve rasgos y F1-Macro. Cada modelo puede elegir su n-grama internamente; reportamos esta elección en la tabla de detalle. El resumen OOF se basa únicamente en los Outer Test del train oficial y no reemplaza al F1 medio por fold.


In [ ]:
df_nested_xgb = df_nested_resumen.copy()
df_nested_xgb["modelo"] = "XGBoost"
columnas = ["variante", "modelo", "f1_train_mean", "f1_nested_mean", "f1_nested_std", "gap_mean"]
df_nested_comparativo = pd.concat(
    [df_nested_lr_rf[columnas], df_nested_xgb[columnas]],
    ignore_index=True,
).sort_values(["variante", "modelo"]).reset_index(drop=True)
print("COMPARACIÓN DE LR, RF Y XGBOOST — SOLO TRAIN OFICIAL")
display(df_nested_comparativo.style.format({
    "f1_train_mean": "{:.4f}", "f1_nested_mean": "{:.4f}",
    "f1_nested_std": "{:.4f}", "gap_mean": "{:.4f}",
}))

# Detalle uniforme para los tres modelos.
detalles = []
for df in resultados_nested_lr_rf.values():
    df = df.copy()
    df["best_ngramas"] = df["best_params"].map(
        lambda p: "uni_bi" if tuple(p["features__tfidf__ngram_range"]) == (1, 2) else "uni"
    )
    detalles.append(df)
for variante in VARIANTES:
    d = nested_detalle[variante].copy()
    d["modelo"] = "XGBoost"
    detalles.append(d)
df_outer_comparativo = pd.concat(detalles, ignore_index=True)
print("\nDETALLE POR OUTER FOLD")
display(df_outer_comparativo[[
    "variante", "modelo", "outer_fold", "best_preprocessing", "best_ngramas",
    "train_f1_macro", "inner_best_f1", "inner_best_std",
    "outer_f1_macro", "generalization_gap", "best_params",
]].sort_values(["variante", "modelo", "outer_fold"]))

print("\nMÉTRICAS POR CLASE EN LOS OUTER FOLDS")
display(df_outer_comparativo[[
    "variante", "modelo", "outer_fold",
    "outer_f1_clase_0", "outer_f1_clase_1", "outer_f1_macro",
]].sort_values(["variante", "modelo", "outer_fold"]).round(4))

# Predicciones OOF: cada fila original debe tener una única predicción externa por modelo.
df_oof = pd.concat(OOF_REGISTROS, ignore_index=True)
resumen_oof = []
for (variante, modelo), grupo in df_oof.groupby(["variante", "modelo"]):
    _, y_ref = cargar_todos_preps(variante, split="train")
    assert len(grupo) == len(y_ref), f"OOF incompleto: {variante}/{modelo}"
    assert grupo["idx_train_oficial"].is_unique, f"OOF duplicado: {variante}/{modelo}"
    g = grupo.sort_values("idx_train_oficial")
    assert np.array_equal(g.y_true.to_numpy(), y_ref.to_numpy()), "Etiquetas OOF desalineadas"
    f1_c0, f1_c1 = f1_score(g.y_true, g.y_pred, labels=[0, 1], average=None, zero_division=0)
    resumen_oof.append({
        "variante": variante, "modelo": modelo,
        "f1_oof_global": f1_score(g.y_true, g.y_pred, average="macro"),
        "f1_oof_clase_0": f1_c0, "f1_oof_clase_1": f1_c1,
    })
df_oof_resumen = pd.DataFrame(resumen_oof)
print("\nRESUMEN OOF (métrica global distinta del promedio de los 5 folds)")
display(df_oof_resumen.sort_values(["variante", "modelo"]).round(4))

print("\nFRECUENCIA DEL PREPROCESAMIENTO GANADOR")
display(pd.crosstab(
    [df_outer_comparativo["variante"], df_outer_comparativo["modelo"]],
    df_outer_comparativo["best_preprocessing"],
))
print("\nFRECUENCIA DEL RANGO DE N-GRAMAS GANADOR")
display(pd.crosstab(
    [df_outer_comparativo["variante"], df_outer_comparativo["modelo"]],
    df_outer_comparativo["best_ngramas"],
))

COMPARACIÓN DE LR, RF Y XGBOOST — SOLO TRAIN OFICIAL


,variante,modelo,f1_train_mean,f1_nested_mean,f1_nested_std,gap_mean
0,cu,LR,0.9107,0.6650,0.0223,0.2457
1,cu,RF,0.8350,0.6650,0.0213,0.1701
2,cu,XGBoost,0.7692,0.6451,0.0057,0.1241
3,es,LR,0.9423,0.7001,0.0120,0.2422
4,es,RF,0.9730,0.6993,0.0272,0.2737
5,es,XGBoost,0.7819,0.6842,0.0180,0.0977
6,mx,LR,0.9074,0.6578,0.0260,0.2496
7,mx,RF,0.8362,0.6322,0.0184,0.2040
8,mx,XGBoost,0.7440,0.6043,0.0342,0.1397



DETALLE POR OUTER FOLD


,variante,modelo,outer_fold,best_preprocessing,best_ngramas,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap,best_params
20,cu,LR,1,lemma,uni_bi,0.992384,0.651933,0.019298,0.665699,0.326684,"{'clf__C': 10, 'clf__solver': 'liblinear', 'fe..."
21,cu,LR,2,lemma,uni,0.889812,0.664332,0.037234,0.627898,0.261914,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
22,cu,LR,3,stem,uni,0.876075,0.659884,0.016365,0.679800,0.196276,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
23,cu,LR,4,stem,uni,0.883730,0.664126,0.035367,0.666857,0.216873,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
24,cu,LR,5,normal,uni,0.911623,0.652990,0.034540,0.684940,0.226684,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
25,cu,RF,1,stem,uni,0.826556,0.651902,0.018307,0.665539,0.161016,"{'clf__max_depth': 10, 'clf__min_samples_split..."
26,cu,RF,2,normal,uni,0.843354,0.663235,0.005178,0.652410,0.190944,"{'clf__max_depth': 20, 'clf__min_samples_split..."
27,cu,RF,3,normal,uni_bi,0.817190,0.660617,0.010574,0.637357,0.179833,"{'clf__max_depth': 10, 'clf__min_samples_split..."
28,cu,RF,4,stem,uni,0.827586,0.667245,0.026400,0.677291,0.150295,"{'clf__max_depth': 10, 'clf__min_samples_split..."
29,cu,RF,5,normal,uni_bi,0.860403,0.650216,0.004637,0.692185,0.168218,"{'clf__max_depth': 20, 'clf__min_samples_split..."



MÉTRICAS POR CLASE EN LOS OUTER FOLDS


...

,variante,modelo,f1_oof_global,f1_oof_clase_0,f1_oof_clase_1
0,cu,LR,0.6654,0.7913,0.5395
1,cu,RF,0.6654,0.8034,0.5274
2,cu,XGBoost,0.6457,0.7689,0.5224
3,es,LR,0.7005,0.7902,0.6108
4,es,RF,0.6997,0.8020,0.5974
5,es,XGBoost,0.6845,0.7826,0.5864
6,mx,LR,0.6584,0.7709,0.5460
7,mx,RF,0.6334,0.7594,0.5073
8,mx,XGBoost,0.6049,0.7261,0.4838



FRECUENCIA DEL PREPROCESAMIENTO GANADOR


best_preprocessing  lemma  normal  stem
variante modelo                        
cu       LR             2       1     2
         RF             0       3     2
         XGBoost        2       1     2
es       LR             0       2     3
         RF             1       1     3
         XGBoost        1       2     2
mx       LR             2       1     2
         RF             0       3     2
         XGBoost        0       2     3


FRECUENCIA DEL RANGO DE N-GRAMAS GANADOR


best_ngramas      uni  uni_bi
variante modelo              
cu       LR         4       1
         RF         3       2
         XGBoost    3       2
es       LR         3       2
         RF         4       1
         XGBoost    4       1
mx       LR         4       1
         RF         5       0
         XGBoost    5       0

### 6.1 Diagnóstico de hiperparámetros de XGBoost

Se informa qué proporción de los 5 Outer Folds seleccionó cada límite de los rangos explorados. Los límites repetidos son una **pista**, no una demostración, de que sería útil explorar otros intervalos. También se guarda cada trial para analizar la trayectoria de Optuna.

In [ ]:
LIMITES_XGB = {
    "n_estimators": (150, 700), "max_depth": (2, 7),
    "learning_rate": (0.01, 0.20), "min_child_weight": (2, 15),
    "subsample": (0.55, 0.90), "colsample_bytree": (0.50, 0.90),
    "gamma": (0.5, 8.0), "reg_alpha": (1e-3, 10.0),
    "reg_lambda": (0.1, 20.0), "scale_pos_weight": (1.0, 2.5),
}
filas_limites = []
for variante in VARIANTES:
    params_por_fold = nested_detalle[variante]["best_params"].tolist()
    for nombre, (lo, hi) in LIMITES_XGB.items():
        valores = [p[nombre] for p in params_por_fold]
        # Algunos parámetros log-uniformes rara vez coinciden con el límite exacto.
        amplitud = hi - lo
        cerca_bajo = sum(v <= lo + 0.02 * amplitud for v in valores)
        cerca_alto = sum(v >= hi - 0.02 * amplitud for v in valores)
        filas_limites.append({
            "variante": variante, "parametro": nombre,
            "min_rango": lo, "max_rango": hi,
            "cerca_min_5_folds": cerca_bajo, "cerca_max_5_folds": cerca_alto,
            "valores_elegidos": valores,
        })
df_diagnostico_limites = pd.DataFrame(filas_limites)
display(df_diagnostico_limites)

# ¿Cuántos de los 50 trials realmente evaluaron cada preprocessing?
conteos_trials = []
for variante in VARIANTES:
    for outer_fold, study in nested_studies[variante].items():
        for prep in PREPROCESAMIENTOS:
            t_ok = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
                    and t.params.get("preprocessing") == prep]
            conteos_trials.append({"variante": variante, "outer_fold": outer_fold,
                                   "preprocessing": prep, "trials_completados": len(t_ok)})
df_conteos_trials = pd.DataFrame(conteos_trials)
print("\nDISTRIBUCIÓN DE TRIALS POR PREPROCESSING")
display(df_conteos_trials)
conteos_trials_ngramas = []
for variante in VARIANTES:
    for outer_fold, study in nested_studies[variante].items():
        for ngramas in NGRAMAS_OPCIONES:
            t_ok = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
                    and t.params.get("ngramas") == ngramas]
            conteos_trials_ngramas.append({
                "variante": variante, "outer_fold": outer_fold,
                "ngramas": ngramas, "trials_completados": len(t_ok),
            })
df_conteos_trials_ngramas = pd.DataFrame(conteos_trials_ngramas)
print("\nDISTRIBUCIÓN DE TRIALS XGBOOST POR N-GRAMAS")
display(df_conteos_trials_ngramas)

,variante,parametro,min_rango,max_rango,cerca_min_5_folds,cerca_max_5_folds,valores_elegidos
0,mx,n_estimators,150.000,700.0,0,0,"[400, 500, 650, 400, 350]"
1,mx,max_depth,2.000,7.0,2,0,"[6, 3, 6, 2, 2]"
2,mx,learning_rate,0.010,0.2,0,0,"[0.043639689845318415, 0.0553808210147886, 0.0..."
3,mx,min_child_weight,2.000,15.0,2,0,"[2, 3, 2, 3, 13]"
4,mx,subsample,0.550,0.9,0,0,"[0.7148569283275314, 0.6705146951557133, 0.644..."
5,mx,colsample_bytree,0.500,0.9,0,0,"[0.6955032024403796, 0.6183002580460784, 0.591..."
6,mx,gamma,0.500,8.0,0,0,"[1.5856736251928039, 0.764968271070108, 1.3070..."
7,mx,reg_alpha,0.001,10.0,3,0,"[0.19003802367439404, 0.9931249154773975, 0.00..."
8,mx,reg_lambda,0.100,20.0,3,0,"[0.6190863822589344, 0.1253906180014041, 0.687..."
9,mx,scale_pos_weight,1.000,2.5,0,0,"[1.9579296055513413, 1.661750925937473, 1.6753..."



DISTRIBUCIÓN DE TRIALS POR PREPROCESSING


,variante,outer_fold,preprocessing,trials_completados
0,mx,1,normal,2
1,mx,1,stem,43
2,mx,1,lemma,5
3,mx,2,normal,3
4,mx,2,stem,44
5,mx,2,lemma,3
6,mx,3,normal,31
7,mx,3,stem,17
8,mx,3,lemma,2
9,mx,4,normal,2



DISTRIBUCIÓN DE TRIALS XGBOOST POR N-GRAMAS


,variante,outer_fold,ngramas,trials_completados
0,mx,1,uni,44
1,mx,1,uni_bi,6
2,mx,2,uni,46
3,mx,2,uni_bi,4
4,mx,3,uni,45
5,mx,3,uni_bi,5
6,mx,4,uni,41
7,mx,4,uni_bi,9
8,mx,5,uni,42
9,mx,5,uni_bi,8


### 6.2 Guardar los resultados de desarrollo — sin tocar test

Esta corrida usa un nuevo directorio `resultados_ml_optuna_tfidf_original_lr_rf` para no sobrescribir la corrida previa. Guarda la selección de n-gramas por fold/trial, la configuración TF-IDF y los resúmenes comparables.


In [ ]:
df_nested_comparativo.to_csv(os.path.join(OUT_DIR, "resumen_nested_ml.csv"), index=False)
df_outer_comparativo.to_csv(os.path.join(OUT_DIR, "outer_folds_ml.csv"), index=False)
df_candidatos_inner_lr_rf.to_csv(os.path.join(OUT_DIR, "candidatos_inner_lr_rf.csv"), index=False)
df_diagnostico_limites.to_csv(os.path.join(OUT_DIR, "diagnostico_limites_xgb.csv"), index=False)
df_conteos_trials.to_csv(os.path.join(OUT_DIR, "conteos_trials_preprocessing_xgb.csv"), index=False)
df_conteos_trials_ngramas.to_csv(os.path.join(OUT_DIR, "conteos_trials_ngramas_xgb.csv"), index=False)
df_oof.to_csv(os.path.join(OUT_DIR, "predicciones_oof_ml.csv"), index=False)
df_oof_resumen.to_csv(os.path.join(OUT_DIR, "resumen_oof_ml.csv"), index=False)
pd.DataFrame([VERSIONES]).to_csv(os.path.join(OUT_DIR, "versiones_paquetes.csv"), index=False)

for variante in VARIANTES:
    for outer_fold, study in nested_studies[variante].items():
        study.trials_dataframe().to_csv(
            os.path.join(OUT_DIR, f"optuna_xgb_{variante}_outer{outer_fold}.csv"),
            index=False,
        )

with open(os.path.join(OUT_DIR, "config_experimento.json"), "w", encoding="utf-8") as fh:
    json.dump({
        "random_state": RANDOM_STATE, "variantes": VARIANTES,
        "tfidf_comun": True, "tfidf_perfil": TFIDF_PERFIL,
        "splitter": "StratifiedGroupKFold", "group_key": "transitive_normal_stem_lemma_message_exact",
        "group_audit": df_auditoria_grupos.to_dict(orient="records"),
        "n_outer": N_OUTER, "n_inner": N_INNER,
        "n_trials_nested_xgb": N_TRIALS_NESTED,
        "n_trials_final_xgb": N_TRIALS_FINAL,
        "grids_lr_rf": {m: grid_lr_rf(m) for m in MODELOS_LR_RF},
        "tfidf_config": {"ngram_range_candidates": [[1, 1], [1, 2]],
                          "max_features": 10000, "min_df": 1, "max_df": 1.0,
                          "sublinear_tf": False, "lowercase": True,
                          "stop_words": "NLTK spanish", "n_stopwords": len(STOP_WORDS)},
        "xgb_search": "TPE seeded; 6 initial prep × ngram trials; total budget includes them",
        "versions": VERSIONES,
    }, fh, indent=2, default=str, ensure_ascii=False)
print(f"Resultados de desarrollo guardados en {OUT_DIR}")

Resultados de desarrollo guardados en ../data\resultados_ml_groupcv_tfidf_lr_rf_10k


## 7. Búsqueda final independiente — todo TRAIN, test cerrado

LR/RF repiten GridSearchCV sobre todo el train seleccionando preprocesamiento, n-gramas e hiperparámetros. XGBoost repite una búsqueda Optuna de 50 trials con las mismas opciones de TF-IDF original LR/RF. Se entrenan y guardan los nueve modelos sobre todo el train oficial. Esta segunda búsqueda **no continúa** los parámetros de los Outer folds.


In [ ]:
FINAL_LR_RF = {}
filas_finales_lr_rf = []
for variante in VARIANTES:
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    final_splits = crear_splits_final(y, grupos_train(variante), variante)
    for modelo in MODELOS_LR_RF:
        mejor, candidatos = buscar_lr_rf_train(X_por_prep, y, modelo, final_splits)
        FINAL_LR_RF[(variante, modelo)] = mejor
        filas_finales_lr_rf.append({
            "variante": variante, "modelo": modelo,
            "preprocessing": mejor["prep"],
            "ngramas": "uni_bi" if tuple(mejor["params"]["features__tfidf__ngram_range"]) == (1, 2) else "uni",
            "best_f1_cv_final": mejor["inner_f1"],
            "cv_std_final": mejor["inner_std"],
            "best_params": mejor["params"],
        })

df_finales_lr_rf = pd.DataFrame(filas_finales_lr_rf)
print("SELECCIÓN FINAL LR/RF — TEST AÚN NO CARGADO")
display(df_finales_lr_rf)

SELECCIÓN FINAL LR/RF — TEST AÚN NO CARGADO


,variante,modelo,preprocessing,ngramas,best_f1_cv_final,cv_std_final,best_params
0,mx,LR,normal,uni,0.647326,0.020884,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
1,mx,RF,stem,uni,0.638092,0.012468,"{'clf__max_depth': 20, 'clf__min_samples_split..."
2,es,LR,normal,uni_bi,0.704801,0.010385,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
3,es,RF,normal,uni,0.702459,0.025208,"{'clf__max_depth': None, 'clf__min_samples_spl..."
4,cu,LR,stem,uni,0.673331,0.012821,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
5,cu,RF,stem,uni,0.674909,0.010114,"{'clf__max_depth': 10, 'clf__min_samples_split..."


In [ ]:
def optimizar_final_variante(variante, n_trials=N_TRIALS_FINAL):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    final_splits = crear_splits_final(y, grupos_train(variante), variante)

    def objective(trial):
        prep = trial.suggest_categorical(
            "preprocessing", list(PREPROCESAMIENTOS.keys())
        )
        ngramas = trial.suggest_categorical("ngramas", list(NGRAMAS_OPCIONES))
        params = sugerir_params_xgb(trial)
        pipeline = crear_pipeline(params, random_state=RANDOM_STATE, ngramas=ngramas)

        scores = cross_val_score(
            estimator=pipeline,
            X=X_por_prep[prep],
            y=y,
            cv=final_splits,
            scoring="f1_macro",
            n_jobs=1,
        )
        trial.set_user_attr("cv_f1_std", float(scores.std(ddof=1)))
        return float(scores.mean())

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    semillas = [(prep, ng) for prep in PREPROCESAMIENTOS for ng in NGRAMAS_OPCIONES]
    if n_trials >= len(semillas):
        for prep_inicial, ngramas_inicial in semillas:
            study.enqueue_trial({"preprocessing": prep_inicial, "ngramas": ngramas_inicial})
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )
    return study


studies_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"OPTIMIZACIÓN FINAL — {variante.upper()}")
    print(f"{'='*70}")

    study = optimizar_final_variante(
        variante, n_trials=N_TRIALS_FINAL
    )
    studies_finales[variante] = study

    print(f"Mejor F1-CV: {study.best_value:.4f}")
    print(f"Mejor preprocessing: {study.best_params['preprocessing']}")
    print(f"Mejor n-gramas: {study.best_params['ngramas']} → {NGRAMAS_OPCIONES[study.best_params['ngramas']]}")
    print("Mejores hiperparámetros XGBoost:")
    for k, v in study.best_params.items():
        if k != "preprocessing":
            print(f"  {k}: {repr(v)}")


OPTIMIZACIÓN FINAL — MX


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.6234
Mejor preprocessing: normal
Mejor n-gramas: uni → (1, 1)
Mejores hiperparámetros XGBoost:
  ngramas: 'uni'
  n_estimators: 400
  max_depth: 4
  learning_rate: 0.07808843841112374
  min_child_weight: 3
  subsample: 0.6564543090657146
  colsample_bytree: 0.5401533888592551
  gamma: 3.8997655003716494
  reg_alpha: 0.21295405833981548
  reg_lambda: 1.4047151878910213
  scale_pos_weight: 1.6619616690238574

OPTIMIZACIÓN FINAL — ES


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.6876
Mejor preprocessing: stem
Mejor n-gramas: uni → (1, 1)
Mejores hiperparámetros XGBoost:
  ngramas: 'uni'
  n_estimators: 550
  max_depth: 5
  learning_rate: 0.020068952882495654
  min_child_weight: 4
  subsample: 0.594236080768487
  colsample_bytree: 0.6904688140281144
  gamma: 5.53036467131824
  reg_alpha: 0.7608383071249113
  reg_lambda: 8.022860715303645
  scale_pos_weight: 1.7671725990532967

OPTIMIZACIÓN FINAL — CU


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.6553
Mejor preprocessing: lemma
Mejor n-gramas: uni_bi → (1, 2)
Mejores hiperparámetros XGBoost:
  ngramas: 'uni_bi'
  n_estimators: 400
  max_depth: 4
  learning_rate: 0.08850871799156305
  min_child_weight: 2
  subsample: 0.6218808921835765
  colsample_bytree: 0.5139346296998972
  gamma: 5.608830546862799
  reg_alpha: 0.13108828799174377
  reg_lambda: 0.5994882945388441
  scale_pos_weight: 2.1536941744392175


In [ ]:
# Resumen de la selección final

filas = []
for variante, study in studies_finales.items():
    fila = {
        "variante": variante,
        "best_f1_cv_final": study.best_value,
        "cv_std_final": study.best_trial.user_attrs.get("cv_f1_std", np.nan),
    }
    fila.update(study.best_params)
    filas.append(fila)


df_params_finales = pd.DataFrame(filas)

display(
    df_params_finales.style.format({
        "best_f1_cv_final": "{:.4f}",
        "cv_std_final": "{:.4f}",
        "learning_rate": "{:.6f}",
        "subsample": "{:.4f}",
        "colsample_bytree": "{:.4f}",
        "gamma": "{:.4f}",
        "reg_alpha": "{:.6f}",
        "reg_lambda": "{:.6f}",
        "scale_pos_weight": "{:.4f}",
    })
)

,variante,best_f1_cv_final,cv_std_final,preprocessing,ngramas,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight
0,mx,0.6234,0.0127,normal,uni,400,4,0.078088,3,0.6565,0.5402,3.8998,0.212954,1.404715,1.6620
1,es,0.6876,0.0098,stem,uni,550,5,0.020069,4,0.5942,0.6905,5.5304,0.760838,8.022861,1.7672
2,cu,0.6553,0.0102,lemma,uni_bi,400,4,0.088509,2,0.6219,0.5139,5.6088,0.131088,0.599488,2.1537


### 7.1 Consolidar y guardar las selecciones finales

Los tres modelos han cerrado la búsqueda sobre todo el train. Esta celda guarda las configuraciones finales sin consultar el test oficial.

In [ ]:
filas_xgb_final = []
for variante, study in studies_finales.items():
    filas_xgb_final.append({
        "variante": variante, "modelo": "XGBoost",
        "preprocessing": study.best_params["preprocessing"],
        "ngramas": study.best_params["ngramas"],
        "best_f1_cv_final": study.best_value,
        "cv_std_final": study.best_trial.user_attrs.get("cv_f1_std", np.nan),
        "best_params": {k: v for k, v in study.best_params.items() if k not in {"preprocessing", "ngramas"}},
    })
df_seleccion_final = pd.concat([
    df_finales_lr_rf, pd.DataFrame(filas_xgb_final),
], ignore_index=True).sort_values(["variante", "modelo"])
print("CONFIGURACIONES FINALES — NUEVE MODELOS, SIN TEST; n-gramas seleccionados por Inner CV")
display(df_seleccion_final)
df_seleccion_final.to_csv(os.path.join(OUT_DIR, "seleccion_final_ml.csv"), index=False)

# Uniformización final: todos los estimadores se entrenan sobre TODO train sin abrir test.
MODELOS_FINALES = {}
for (variante, modelo), mejor in FINAL_LR_RF.items():
    estimador = mejor["estimator"]  # Ya está refit sobre train por GridSearchCV.
    MODELOS_FINALES[(variante, modelo)] = estimador
    joblib.dump(estimador, os.path.join(MODELS_DIR,
                f"{modelo.lower()}_{variante}_{mejor['prep']}.pkl"))
for variante, study in studies_finales.items():
    best_all = study.best_params.copy()
    prep = best_all.pop("preprocessing")
    ngramas = best_all.pop("ngramas")
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    estimador = crear_pipeline(best_all, random_state=RANDOM_STATE, ngramas=ngramas)
    estimador.fit(X_por_prep[prep], y)
    MODELOS_FINALES[(variante, "XGBoost")] = estimador
    joblib.dump(estimador, os.path.join(MODELS_DIR, f"xgboost_{variante}_{prep}.pkl"))
print("Modelos finales entrenados y guardados SOLO con train:", len(MODELOS_FINALES))
for variante, study in studies_finales.items():
    study.trials_dataframe().to_csv(
        os.path.join(OUT_DIR, f"optuna_xgb_final_{variante}.csv"),
        index=False,
    )

CONFIGURACIONES FINALES — NUEVE MODELOS, SIN TEST; n-gramas seleccionados por Inner CV


,variante,modelo,preprocessing,ngramas,best_f1_cv_final,cv_std_final,best_params
4,cu,LR,stem,uni,0.673331,0.012821,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
5,cu,RF,stem,uni,0.674909,0.010114,"{'clf__max_depth': 10, 'clf__min_samples_split..."
8,cu,XGBoost,lemma,uni_bi,0.655284,0.010170,"{'n_estimators': 400, 'max_depth': 4, 'learnin..."
2,es,LR,normal,uni_bi,0.704801,0.010385,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
3,es,RF,normal,uni,0.702459,0.025208,"{'clf__max_depth': None, 'clf__min_samples_spl..."
7,es,XGBoost,stem,uni,0.687568,0.009827,"{'n_estimators': 550, 'max_depth': 5, 'learnin..."
0,mx,LR,normal,uni,0.647326,0.020884,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'f..."
1,mx,RF,stem,uni,0.638092,0.012468,"{'clf__max_depth': 20, 'clf__min_samples_split..."
6,mx,XGBoost,normal,uni,0.623387,0.012694,"{'n_estimators': 400, 'max_depth': 4, 'learnin..."


Modelos finales entrenados y guardados SOLO con train: 9


## 8. TEST OFICIAL (bloqueado por defecto)

Solo habilitar `EJECUTAR_TEST_OFICIAL=True` en la configuración **una vez cerrados** los hiperparámetros, TF-IDF, método de selección, análisis sobre development y modelos finales. La celda reutiliza los nueve modelos ya ajustados: no realiza ninguna optimización ni entrenamiento nuevo. La evaluación de test oficial debe hacerse una sola vez.

In [ ]:
RESULTADOS_TEST = []
PREDICCIONES_TEST = {}

if not EJECUTAR_TEST_OFICIAL:
    print(
        "TEST OFICIAL BLOQUEADO: "
        "los modelos ya fueron seleccionados y ajustados sin abrir el test."
    )
else:
    for _, fila in df_seleccion_final.sort_values(
        ["variante", "modelo"]
    ).iterrows():

        variante = fila["variante"]
        modelo = fila["modelo"]
        prep = fila["preprocessing"]

        estimador = MODELOS_FINALES[(variante, modelo)]

        # El test se abre únicamente en la evaluación final.
        df_test = cargar_split(
            variante,
            prep,
            split="test",
        )

        X_test, y_test = preparar_xy(df_test)
        pred = estimador.predict(X_test)

        ref_cv = df_nested_comparativo.query(
            "variante == @variante and modelo == @modelo"
        ).iloc[0]

        RESULTADOS_TEST.append({
            "variante": variante,
            "modelo": modelo,
            "preprocessing": prep,
            "outer_f1_mean": ref_cv["f1_nested_mean"],
            "outer_f1_std": ref_cv["f1_nested_std"],
            "f1_cv_final": fila["best_f1_cv_final"],
            "f1_macro_test": f1_score(
                y_test, pred, average="macro"
            ),
            "accuracy_test": accuracy_score(y_test, pred),
            "precision_macro_test": precision_score(
                y_test, pred, average="macro", zero_division=0
            ),
            "recall_macro_test": recall_score(
                y_test, pred, average="macro", zero_division=0
            ),
            "f1_clase_0": f1_score(
                y_test, pred, pos_label=0, zero_division=0
            ),
            "f1_clase_1": f1_score(
                y_test, pred, pos_label=1, zero_division=0
            ),
        })

        PREDICCIONES_TEST[(variante, modelo)] = (
            y_test,
            pred,
            df_test,
        )

        print(
            f"{variante.upper()} | {modelo} | {prep}: "
            f"TEST F1-Macro="
            f"{RESULTADOS_TEST[-1]['f1_macro_test']:.4f}"
        )

    df_resultados_test = pd.DataFrame(RESULTADOS_TEST)

    display(
        df_resultados_test
        .sort_values(["variante", "modelo"])
        .round(4)
    )

    df_resultados_test.to_csv(
        os.path.join(OUT_DIR, "resultados_test_ml.csv"),
        index=False,
    )

    print(
        "EVALUACIÓN FINAL TERMINADA: "
        "no volver a seleccionar hiperparámetros en función del test."
    )


TEST OFICIAL BLOQUEADO: los modelos ya fueron seleccionados y ajustados sin abrir el test.


### 8.1 Análisis de errores en los nueve modelos (solo cuando test esté habilitado)

El reporte por clase y la matriz de confusión se presentan para cada variante y modelo. No deben utilizarse para seguir modificando el procedimiento y escoger una nueva versión que optimice este mismo test.

In [ ]:
if not EJECUTAR_TEST_OFICIAL:
    print("Análisis de errores no disponible: test oficial bloqueado.")
else:
    for (variante, modelo), (y_test, pred, df_test) in PREDICCIONES_TEST.items():
        print(f"\n{'='*60}\n{variante.upper()} / {modelo}")
        print(classification_report(y_test, pred, digits=4, zero_division=0))
        print("Matriz de confusión:")
        print(confusion_matrix(y_test, pred))

Análisis de errores no disponible: test oficial bloqueado.


### 8.2 Contribución de las nueve características lingüísticas (exploratorio)

Los nueve modelos finales ya están ajustados con el train oficial, aun con el test deshabilitado. Por ello es posible estudiar sus atributos sin consultar el test. **No compares directamente la magnitud de un coeficiente de LR con la importancia de RF/XGBoost**: son medidas de naturaleza distinta. No utilices estos resultados para reajustar un modelo y luego presentar el mismo test como independiente si ya lo evaluaste.

In [ ]:
def importancia_linguistica(estimador, modelo):
    paso_clf = "xgb" if modelo == "XGBoost" else "clf"
    clf = estimador.named_steps[paso_clf]
    if hasattr(clf, "feature_importances_"):
        importancia = np.asarray(clf.feature_importances_)
    elif hasattr(clf, "coef_"):
        importancia = np.abs(clf.coef_[0])
    else:
        return None
    preproc = estimador.named_steps["features"]
    n_tfidf = len(preproc.named_transformers_["tfidf"].get_feature_names_out())
    ling = importancia[n_tfidf:n_tfidf + len(FEATURE_COLS)]
    if len(ling) != len(FEATURE_COLS):
        return None
    return dict(zip(FEATURE_COLS, ling))

filas = []
for (variante, modelo), estimador in MODELOS_FINALES.items():
    imp = importancia_linguistica(estimador, modelo)
    if imp is not None:
        for nombre, valor in imp.items():
            filas.append({
                "variante": variante, "modelo": modelo,
                "caracteristica": nombre, "importancia": valor,
            })
df_importancias_linguisticas = pd.DataFrame(filas)
display(df_importancias_linguisticas)
df_importancias_linguisticas.to_csv(
    os.path.join(OUT_DIR, "importancias_linguisticas_ml.csv"), index=False
)

,variante,modelo,caracteristica,importancia
0,mx,LR,n_exc,0.090699
1,mx,LR,n_int,0.020191
2,mx,LR,n_may,0.043610
3,mx,LR,n_emo,0.123889
4,mx,LR,n_ris,0.531204
...,...,...,...,...
76,cu,XGBoost,n_ris,0.003521
77,cu,XGBoost,n_neg,0.004186
78,cu,XGBoost,n_elo,0.013808
79,cu,XGBoost,n_com,0.005298


## Lectura e interpretación de esta corrida agrupada

1. Los Outer e Inner folds están **agrupados**; no se comparte entre train/validación ningún grupo conectado por coincidencia textual en `normal`, `stem`, `lemma` o mensaje original.
2. La validación utiliza **5 Outer × 3 Inner** con `StratifiedGroupKFold`.
3. La descontaminación Train ↔ Test ya fue resuelta una sola vez en `01_preprocesamiento_ML_CORREGIDO.ipynb`; este notebook no vuelve a eliminar registros.
4. Los grupos con etiquetas mixtas se mantienen completos en un único fold y se registran como ruido/ambigüedad de anotación.
5. `F1 Train`, `F1 Outer`, resultados OOF, F1 por clase y resultados por Outer Fold se calculan sobre el procedimiento agrupado.
6. El test oficial permanece bloqueado hasta la evaluación final.
7. Los notebooks 10k y 20k usan el mismo dataset final, grupos, semillas, clasificadores y protocolo 5×3. La diferencia experimental corresponde al perfil TF-IDF de cada notebook.
